# PoolPy Decoder - User-Friendly Interface

This notebook provides an easy-to-use interface for decoding readouts using the PoolPy decoder.

## Instructions:
1. **Cell 1**: Install dependencies from requirements.txt
2. **Cell 2**: Import required libraries
3. **Cell 3**: Define your decoder configuration (paths, parameters)
4. **Cell 4**: Run the decoder
5. **Cell 5**: View and download results

Simply fill in the configuration variables in Cell 3 and run the cells in order.

## Decoder types

The decoder auto infers the type of readout returns a type alongside the positive samples. This characteristic has four possible distinct values:

- **Continuous**: Used if the readout is inferred to be continuous and the single well value is inferred via penalized regression.
- **Unique**: Used if the readout is binary and there exists only one set of positive samples (of maximum *differentiate* samples) that can provide the measured readout.
- **Multiple**: Used if the readout is binary and there exists multiple sets of positive samples (of maximum *differentiate* samples) that can provide the measured readout.
- **Putative**: Used if the readout is binary and there exists more sets of positive samples (of maximum *differentiate* samples) that can provide the measured readout than the number of unique samples across all sets.


In [1]:
import subprocess
import sys
import os

# Setup environment from requirements.txt
print("Environment Setup")
print("=" * 70)

# Get the working directory
working_dir = os.getcwd()
print(f"Working directory: {working_dir}")
print()

requirements_path = os.path.join(working_dir, "requirements.txt")

# Try to read requirements.txt to show what packages are needed
try:
    with open(requirements_path, 'r') as f:
        requirements = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    
    print(f"✓ Requirements file found: {requirements_path}")
    print(f"\nRequired packages ({len(requirements)} total):")
    for req in requirements:
        print(f"  - {req}")
    
    print("\n" + "-" * 70)
    print("Environment Status:")
    print(f"  Python version: {sys.version}")
    print(f"  Python executable: {sys.executable}")
    
    # Check if key packages are available
    key_packages = ['pandas', 'numpy', 'scipy', 'sklearn']
    for pkg_name in key_packages:
        try:
            __import__(pkg_name)
            print(f"  ✓ {pkg_name} is installed")
        except ImportError:
            print(f"  ✗ {pkg_name} is NOT installed")
    
except FileNotFoundError:
    print(f"✗ Requirements file not found: {requirements_path}")
    print("Please ensure requirements.txt exists in the working directory.")

print("=" * 70)
print("\nNote: For UV environments, dependencies are typically pre-installed.")
print("If you need to install packages, use: uv pip install -r requirements.txt")


Environment Setup
Working directory: /Users/ltalamanca/My Drive/Git/PoolPy

✓ Requirements file found: /Users/ltalamanca/My Drive/Git/PoolPy/requirements.txt

Required packages (50 total):
  - appnope==0.1.4
  - asttokens==3.0.0
  - cmcrameri==1.9
  - comm==0.2.2
  - contourpy==1.3.2
  - cycler==0.12.1
  - debugpy==1.8.14
  - decorator==5.2.1
  - et-xmlfile==2.0.0
  - executing==2.2.0
  - fonttools==4.57.0
  - ipykernel==6.29.5
  - ipython==9.2.0
  - ipython-pygments-lexers==1.1.1
  - jedi==0.19.2
  - jupyter-client==8.6.3
  - jupyter-core==5.7.2
  - kiwisolver==1.4.8
  - matplotlib==3.7.3
  - matplotlib-inline==0.1.7
  - nest-asyncio==1.6.0
  - numpy==1.26.4
  - openpyxl==3.1.5
  - packaging==25.0
  - pandas==2.2.3
  - parso==0.8.4
  - pexpect==4.9.0
  - pillow==11.2.1
  - platformdirs==4.3.7
  - prompt-toolkit==3.0.51
  - psutil==7.0.0
  - ptyprocess==0.7.0
  - pure-eval==0.2.3
  - pygments==2.19.1
  - pyparsing==3.2.3
  - python-dateutil==2.9.0.post0
  - pytz==2025.2
  - pyzmq==26.4

In [2]:
import pandas as pd
import numpy as np
import subprocess
import os
from pathlib import Path

print("✓ All libraries imported successfully!")
print(f"\nPython executable: {sys.executable}")
print(f"Working directory: {os.getcwd()}")


✓ All libraries imported successfully!

Python executable: /Users/ltalamanca/uv_2/bin/python
Working directory: /Users/ltalamanca/My Drive/Git/PoolPy


## Configuration

Edit the variables below to specify your decoder settings. By default, the notebook now uses the test folder in this workspace.

Parameters:
- path_to_WA: Path to the well assignment (WA) matrix CSV file (defaults to working_dir/test/test_wa_notebook.csv)
- readout: Path to the readout CSV file, binary or continuous (defaults to working_dir/test/test_continuous_notebook.csv)
- differentiate: Maximum number of expected active compounds (for sparsity control), default: 2
- diluting: Whether to use dilution scaling in continuous decoding (True/False), default: True
- min_signal: Optional minimum signal threshold (set to None to skip), default: None
- output_dir: Directory where decoded results will be saved (defaults to working_dir/test)
- python_executable: Python executable to use (automatically uses the current Python kernel)

To use custom paths:
Simply edit the lines in Cell 5, for example:
```python
path_to_WA = os.path.join(working_dir, "path/to/my_wa_matrix.csv")
readout = os.path.join(working_dir, "data/my_readout.csv")
```
The decoder auto-infers if the readout is binary or continous.

In [3]:
# ============================================================================
# USER CONFIGURATION - EDIT THESE VARIABLES
# ============================================================================

# Get the working directory (set automatically)
working_dir = os.getcwd()

# Default test folder containing decoder and sample files
test_dir = os.path.join(working_dir, "test")

# Path to the well assignment (WA) matrix CSV file
path_to_WA = os.path.join(test_dir, "test_wa_notebook.csv")

# Path to the readout CSV file (binary or continuous format)
readout = os.path.join(test_dir, "test_continuous_notebook.csv")

# Maximum number of expected postive samples
differentiate = 2

# Whether to use dilution scaling in continuous decoding (True/False)
diluting = True

# Optional minimum signal threshold (set to None to skip)
min_signal = None

# Continuous decoder hyperparameters (explicitly passed to decode_N.py)
# alpha: regularization strength (higher => stronger regularization)
alpha = 0.05
# l1_ratio: 1.0 = Lasso, 0.0 = Ridge, values in between = Elastic-Net style
l1_ratio = 1

# Output directory for results (defaults to test folder)
output_dir = test_dir

# Python executable (uses the current Python that's running the notebook)
python_executable = sys.executable

# ============================================================================
# Validation: Check if input files exist
# ============================================================================

print("Configuration:")
print("=" * 70)
print(f"Working Directory: {working_dir}")
print(f"Test Directory:    {test_dir}")
print(f"WA Matrix:         {os.path.basename(path_to_WA)}")
print(f"Readout CSV:       {os.path.basename(readout)}")
print(f"Differentiate:     {differentiate}")
print(f"Diluting:          {diluting}")
print(f"Min Signal:        {min_signal}")
print(f"Alpha:             {alpha}")
print(f"L1 Ratio:          {l1_ratio}")
print(f"Output Directory:  {output_dir}")
print(f"Python Executable: {python_executable}")
print("=" * 70)

# Check if files exist
print("\nValidation:")
if os.path.exists(path_to_WA):
    print(f"✓ WA Matrix file found: {os.path.basename(path_to_WA)}")
else:
    print(f"✗ WA Matrix file NOT found: {os.path.basename(path_to_WA)}")
    print(f"  Looking in: {path_to_WA}")

if os.path.exists(readout):
    print(f"✓ Readout file found: {os.path.basename(readout)}")
else:
    print(f"✗ Readout file NOT found: {os.path.basename(readout)}")
    print(f"  Looking in: {readout}")

if os.path.exists(output_dir):
    print(f"✓ Output directory exists: {output_dir}")
else:
    print(f"✗ Output directory does NOT exist: {output_dir}")

print(f"✓ Python executable: {python_executable}")

print("=" * 70)

Configuration:
Working Directory: /Users/ltalamanca/My Drive/Git/PoolPy
Test Directory:    /Users/ltalamanca/My Drive/Git/PoolPy/test
WA Matrix:         test_wa_notebook.csv
Readout CSV:       test_continuous_notebook.csv
Differentiate:     2
Diluting:          True
Min Signal:        None
Alpha:             0.05
L1 Ratio:          1
Output Directory:  /Users/ltalamanca/My Drive/Git/PoolPy/test
Python Executable: /Users/ltalamanca/uv_2/bin/python

Validation:
✓ WA Matrix file found: test_wa_notebook.csv
✓ Readout file found: test_continuous_notebook.csv
✓ Output directory exists: /Users/ltalamanca/My Drive/Git/PoolPy/test
✓ Python executable: /Users/ltalamanca/uv_2/bin/python


## Execute Decoder

Run the decoder with the configuration settings from the previous cell.

This interface explicitly passes continuous decoder hyperparameters:
- alpha (default 0.05)
- l1_ratio (default 1)

Change them in the configuration cell before running this step.

In [7]:
# Use configured Python executable, fallback to system Python
exec_python = python_executable if os.path.exists(python_executable) else sys.executable

# Path to decode_N.py script (kept in project root)
script_path = os.path.join(working_dir, "decode_N.py")

if not os.path.exists(script_path):
    print(f"Error: decode_N.py not found at {script_path}")
    print(f"Expected location: {working_dir}")
else:
    # Build the decoder command
    cmd = [
        exec_python,
        script_path,
        "--path_to_WA", path_to_WA,
        "--readout", readout,
        "--differentiate", str(differentiate),
        "--diluting", "True" if diluting else "False",
        "--alpha", str(alpha),
        "--l1_ratio", str(l1_ratio),
    ]
    
    # Add optional min_signal parameter if provided
    if min_signal is not None:
        cmd.extend(["--min_signal", str(min_signal)])
    
    # Display the command being executed
    print("Executing decoder...")
    print("=" * 70)
    print("Continuous decoder args passed to decode_N.py:")
    print(f"  --alpha {alpha}")
    print(f"  --l1_ratio {l1_ratio}")
    print(f"Command: {' '.join(cmd)}")
    print("=" * 70)
    print()
    
    # Run the decoder
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            cwd=output_dir,
            timeout=300  # 5-minute timeout
        )
        
        print(f"Return code: {result.returncode}")
        print()
        
        if result.stdout:
            print("STDOUT:")
            print(result.stdout)
        
        if result.stderr:
            print("STDERR:")
            print(result.stderr)
        
        # Check for success
        if result.returncode == 0:
            print("\n" + "=" * 70)
            print("✓ DECODING COMPLETED SUCCESSFULLY")
            print("=" * 70)
        else:
            print("\n" + "=" * 70)
            print("✗ DECODING FAILED")
            print("=" * 70)
    
    except subprocess.TimeoutExpired:
        print("✗ Decoder execution timed out (5 minutes)")
    except Exception as e:
        print(f"✗ Error running decoder: {e}")

Executing decoder...
Continuous decoder args passed to decode_N.py:
  --alpha 0.05
  --l1_ratio 1
Command: /Users/ltalamanca/uv_2/bin/python /Users/ltalamanca/My Drive/Git/PoolPy/decode_N.py --path_to_WA /Users/ltalamanca/My Drive/Git/PoolPy/test/test_wa_notebook.csv --readout /Users/ltalamanca/My Drive/Git/PoolPy/test/test_continuous_notebook.csv --differentiate 2 --diluting True --alpha 0.05 --l1_ratio 1

Return code: 0


✓ DECODING COMPLETED SUCCESSFULLY


## View Results

Display and analyze the decoded results from the previous execution.


In [8]:
# Load and display the decoded results
output_csv = os.path.join(output_dir, "decoded_readouts.csv")

if os.path.exists(output_csv):
    print(f"Loading results from: {output_csv}")
    print("=" * 70)
    
    decoded_df = pd.read_csv(output_csv)
    
    print(f"\nDecoded Results (Total: {len(decoded_df)} readouts)")
    print("-" * 70)
    
    # Display the results in a formatted table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    print(decoded_df.to_string(index=False))
    
    print("\n" + "=" * 70)
    print(f"✓ Results file: {output_csv}")
    print("=" * 70)
    
    # Show summary statistics
    print("\nSummary:")
    print(f"  Total readouts: {len(decoded_df)}")
    print('You might need to scroll to the right to see all decoder results')
    if 'Decoded Type' in decoded_df.columns:
        print(f"\n  Decoded Types:")
        for dtype, count in decoded_df['Decoded Type'].value_counts().items():
            print(f"    - {dtype}: {count}")
    
else:
    print(f"✗ Results file not found: {output_csv}")
    print("The decoder may not have run successfully or the output path is incorrect.")


Loading results from: /Users/ltalamanca/My Drive/Git/PoolPy/test/decoded_readouts.csv

Decoded Results (Total: 8 readouts)
----------------------------------------------------------------------
           readout_id Decoded Type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         Decoder Output
         continuous_1   cont

In [6]:
# Test continuous decoding with grid search and shared EN parameters across readouts
import os
import shlex
import subprocess
import pandas as pd

# Reuse paths already defined in notebook; provide sensible defaults if not present
working_dir = globals().get("working_dir", os.getcwd())
script_path = globals().get("script_path", os.path.join(working_dir, "decode_N.py"))
path_to_WA = globals().get("path_to_WA", os.path.join(working_dir, "test", "test_wa_notebook.csv"))
readout_input = globals().get("readout_input", os.path.join(working_dir, "test", "test_continuous_notebook.csv"))
differentiate = globals().get("differentiate", 2)
python_path = globals().get("python_path", "python")
output_dir = globals().get("output_dir", working_dir)

# Grid-search continuous decoder options
grid_objective = "mse"   # options: mse, mae, rmse, r2
force_continuous = True
grid_search = True
return_all_fits = False
save_grid_decoders = True

cmd = [
    python_path,
    script_path,
    "--path_to_WA", path_to_WA,
    "--readout", readout_input,
    "--differentiate", str(differentiate),
    "--force_continuous", str(force_continuous),
    "--grid_search", str(grid_search),
    "--return_all_fits", str(return_all_fits),
    "--save_grid_decoders", str(save_grid_decoders),
    "--grid_objective", grid_objective,
]

print("Running continuous grid-search decoder:")
print(" ".join(shlex.quote(part) for part in cmd))

run_result = subprocess.run(
    cmd,
    cwd=output_dir,
    capture_output=True,
    text=True,
    check=False,
    timeout=300,
 )

print("\nExit code:", run_result.returncode)
if run_result.stdout:
    print("\nSTDOUT:\n" + run_result.stdout.strip())
if run_result.stderr:
    print("\nSTDERR:\n" + run_result.stderr.strip())


grid_csv = os.path.join(output_dir, "decoded_grid_decoders.csv")


if os.path.exists(grid_csv):
    print("\nPreview of decoded_grid_decoders.csv")
    print("-" * 70)
    grid_df = pd.read_csv(grid_csv)
    print(grid_df.head(10).to_string(index=False))
    print(f"\nTotal grid-decoder rows: {len(grid_df)}")
else:
    print(f"\nGrid decoder file not found: {grid_csv}")

Running continuous grid-search decoder:
python '/Users/ltalamanca/My Drive/Git/PoolPy/decode_N.py' --path_to_WA '/Users/ltalamanca/My Drive/Git/PoolPy/test/test_wa_notebook.csv' --readout '/Users/ltalamanca/My Drive/Git/PoolPy/test/test_continuous_notebook.csv' --differentiate 2 --force_continuous True --grid_search True --return_all_fits False --save_grid_decoders True --grid_objective mse

Exit code: 0

STDERR:
/Users/ltalamanca/uv_2/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.294e-03, tolerance: 2.261e-03
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/Users/ltalamanca/uv_2/lib/python3.12/site-packa